# SafeAgent CPU baseline on Kaggle

This notebook launches the committed character TF-IDF and Linear SVM baseline.
It uses the package commands and the frozen Python 3.11 environment. Accelerator
can stay set to None and Internet must be enabled. No shell command from the
dataset is executed.

Follow [the Kaggle guide](https://github.com/taed2-2627q1-gced-upc/taed2-SafeAgent/blob/main/docs/kaggle-training.md)
to configure DVC access and optional shared MLflow logging. Kaggle execution of
this notebook has not been verified. Use a fresh session and Run All after
filling the settings below.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import platform
import re
import shutil
import subprocess
import sys

from kaggle_secrets import BackendError, CredentialError, UserSecretsClient

GIT_COMMIT = ""
KAGGLE_NOTEBOOK_REF = ""
KAGGLE_NOTEBOOK_VERSION = None
RUN_SHARED_EXPERIMENT = False

REPOSITORY = "https://github.com/taed2-2627q1-gced-upc/taed2-SafeAgent.git"
TEMP_ROOT = Path("/kaggle/temp")
PROJECT_ROOT = TEMP_ROOT / "safeagent"
OUTPUT_ROOT = Path("/kaggle/working/safeagent-results")

## Check the session settings

Set `GIT_COMMIT` to the full published source commit containing this notebook and
the baseline code. Set the real notebook reference, such as
`joelmrquezalvarez/<notebook-slug>`, and the positive saved source version shown
by Kaggle. For a new submission, enter its intended next version and confirm
that value on the saved run page afterward. A mismatch needs correction in the
downloaded provenance before using the result as experiment evidence.

Keep `RUN_SHARED_EXPERIMENT` false for development training and validation.
Setting it to true starts one formal MLflow run and uploads the fitted model to
DVC. That path needs both DVC and MLflow access.

In [ ]:
if not re.fullmatch(r"[0-9a-f]{40}", GIT_COMMIT):
    raise ValueError("Set GIT_COMMIT to a full published Git commit")
if not re.fullmatch(r"[a-z0-9_]+/[a-z0-9_-]+", KAGGLE_NOTEBOOK_REF):
    raise ValueError("Set the actual Kaggle notebook reference")
if (
    not isinstance(KAGGLE_NOTEBOOK_VERSION, int)
    or isinstance(KAGGLE_NOTEBOOK_VERSION, bool)
    or KAGGLE_NOTEBOOK_VERSION < 1
):
    raise ValueError("Set the saved Kaggle source version")
if not isinstance(RUN_SHARED_EXPERIMENT, bool):
    raise TypeError("RUN_SHARED_EXPERIMENT must be true or false")
if PROJECT_ROOT.exists() or OUTPUT_ROOT.exists():
    raise RuntimeError("Start a fresh session to preserve existing results")
TEMP_ROOT.mkdir(parents=True, exist_ok=True)
COMMAND_ENV = os.environ.copy()

## Load runtime secrets

Enable `DVC_ACCESS_KEY_ID` and `DVC_SECRET_ACCESS_KEY` under Add-ons > Secrets.
For a shared run, also enable `MLFLOW_TRACKING_USERNAME` and
`MLFLOW_TRACKING_PASSWORD`. Secret values stay in the command environment and
are never written to configuration or result files. Command output is captured
and secret values are removed before display.

In [ ]:
secret_labels = {
    "AWS_ACCESS_KEY_ID": "DVC_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY": "DVC_SECRET_ACCESS_KEY",
}
if RUN_SHARED_EXPERIMENT:
    secret_labels.update({
        "MLFLOW_TRACKING_USERNAME": "MLFLOW_TRACKING_USERNAME",
        "MLFLOW_TRACKING_PASSWORD": "MLFLOW_TRACKING_PASSWORD",
    })
secrets_client = UserSecretsClient()
for variable, label in secret_labels.items():
    try:
        value = secrets_client.get_secret(label)
    except (BackendError, CredentialError):
        raise RuntimeError(f"Enable the Kaggle Secret {label}") from None
    if not value:
        raise ValueError(f"The Kaggle Secret {label} is empty")
    COMMAND_ENV[variable] = value
del value
COMMAND_ENV["MLFLOW_TRACKING_URI"] = (
    "https://dagshub.com/Pau-Balaguer/taed2-SafeAgent.mlflow"
)

The command helper uses argument lists and stops on failure. It captures output
so authentication errors can be displayed without exposing loaded secret values.
The dataset commands themselves are always handled as model inputs.

In [ ]:
def run(*arguments, directory=PROJECT_ROOT):
    result = subprocess.run(
        list(arguments), cwd=directory, env=COMMAND_ENV,
        capture_output=True, text=True, check=False,
    )
    stdout = result.stdout
    stderr = result.stderr
    for secret_variable in secret_labels:
        stdout = stdout.replace(COMMAND_ENV[secret_variable], "[redacted]")
        stderr = stderr.replace(COMMAND_ENV[secret_variable], "[redacted]")
    output = stdout + stderr
    if output.strip():
        print(output.strip())
    if result.returncode:
        raise RuntimeError(f"Command failed: {arguments[0]}") from None
    return stdout.strip()

## Clone and install the frozen environment

The checkout and caches use `/kaggle/temp`, so saved outputs contain only selected
results. The notebook kernel supplies standard Python and Kaggle Secrets.
Every package command runs through uv in the separate project environment.
The Python check below rejects a different project interpreter.

In [ ]:
run("git", "clone", REPOSITORY, str(PROJECT_ROOT), directory=TEMP_ROOT)
run("git", "checkout", "--detach", GIT_COMMIT)
if run("git", "rev-parse", "HEAD") != GIT_COMMIT:
    raise RuntimeError("The checkout differs from the requested commit")
if shutil.which("uv") is None:
    run(sys.executable, "-m", "pip", "install", "uv==0.10.10")
UV = ["uv", "run", "--frozen", "--group", "data"]
run("uv", "sync", "--frozen", "--python", "3.11", "--group", "data")
run(*UV, "python", "-c", (
    "import sys; import taed2_safeagent; "
    "assert sys.version_info[:2] == (3, 11); "
    "print('Project Python', sys.version.split()[0])"
))
run(*UV, "dvc", "config", "--local", "cache.dir", str(TEMP_ROOT / "dvc-cache"))

## Recover and validate the data

DVC recovers the versioned inputs and the data validation target checks them.
The baseline uses 26,660 training and 3,333 validation examples. The 3,332 test
examples stay reserved. The shared package input builder supplies command text,
excluding labels, annotation reason, category and context from the features.
The repository must remain clean before a formal experiment.

In [ ]:
run(*UV, "dvc", "pull")
run(*UV, "dvc", "repro", "validate")
if run("git", "status", "--porcelain"):
    raise RuntimeError("Recovered source or data metadata differs from the commit")
split_report = json.loads((PROJECT_ROOT / "reports/data/split.json").read_text())
expected_rows = {"train": 26660, "validation": 3333, "test": 3332}
if {name: data["rows"] for name, data in split_report["splits"].items()} != expected_rows:
    raise ValueError("The prepared split differs from the approved dataset")
input_lock = (PROJECT_ROOT / "dvc.lock").read_bytes()
session = {
    "git_commit": GIT_COMMIT,
    "input_dvc_lock_sha256": hashlib.sha256(input_lock).hexdigest(),
    "kaggle_notebook_ref": KAGGLE_NOTEBOOK_REF,
    "kaggle_source_version": KAGGLE_NOTEBOOK_VERSION,
    "kaggle_source_version_status": "pending saved run confirmation",
    "shared_experiment": RUN_SHARED_EXPERIMENT,
    "platform": platform.platform(),
    "machine": platform.machine(),
}
print("Prepared rows", expected_rows)

## Run one baseline path

The development path runs training and validation without shared logging.
The formal path runs only the experiment command, which includes training,
validation, DVC model upload and shared MLflow logging. It never follows a
development fit with a second fit. No test evaluation or parameter search occurs.
A command failure stops the notebook.

In [ ]:
if RUN_SHARED_EXPERIMENT:
    run(*UV, "python", "-m", "taed2_safeagent.modeling.experiment", "--params", "params.yaml")
    receipt = json.loads((PROJECT_ROOT / "reports/baseline/mlflow_run.json").read_text())
    if receipt["source_commit"] != GIT_COMMIT:
        raise ValueError("The shared run receipt belongs to another source commit")
    session["mlflow_run_id"] = receipt["run_id"]
    session["mlflow_run_url"] = receipt["run_url"]
else:
    run(*UV, "python", "-m", "taed2_safeagent.modeling.train", "--params", "params.yaml")
    run(*UV, "python", "-m", "taed2_safeagent.modeling.evaluate", "--params", "params.yaml")
metrics = json.loads((PROJECT_ROOT / "reports/baseline/metrics.json").read_text())
print("Validation macro F1", metrics["macro_f1"])
print("DENY recall", metrics["classes"]["DENY"]["recall"])
print("DENY predicted ALLOW", metrics["deny_to_allow_count"])

## Export selected results

Only the model, validation reports, data check reports, settings, input DVC lock and
session provenance are copied to saved outputs. Development results exclude the
old shared run receipt recovered with Git. A formal result includes its new run
receipt. Git publication and any later output commit remain separate steps.
The formal path also saves its output DVC lock. A development fit has not yet
been versioned as a new DVC output.
Confirm the recorded source version against the saved Kaggle run after completion.

In [ ]:
OUTPUT_ROOT.mkdir(parents=True)
for relative in ("models/tfidf_char_linear_svm", "reports/data", "reports/baseline"):
    destination = OUTPUT_ROOT / relative
    shutil.copytree(PROJECT_ROOT / relative, destination)
if not RUN_SHARED_EXPERIMENT:
    (OUTPUT_ROOT / "reports/baseline/mlflow_run.json").unlink(missing_ok=True)
shutil.copy2(PROJECT_ROOT / "params.yaml", OUTPUT_ROOT / "params.yaml")
(OUTPUT_ROOT / "input_dvc.lock").write_bytes(input_lock)
if RUN_SHARED_EXPERIMENT:
    version_path = (PROJECT_ROOT / receipt["model_dvc_path"]).resolve()
    project_path = PROJECT_ROOT.resolve()
    if (
        not version_path.is_relative_to(project_path)
        or version_path.suffix not in {".dvc", ".lock"}
    ):
        raise ValueError("The model version metadata path is invalid")
    version_output = OUTPUT_ROOT / version_path.relative_to(project_path)
    version_output.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(version_path, version_output)
    shutil.copy2(PROJECT_ROOT / "dvc.lock", OUTPUT_ROOT / "output_dvc.lock")
(OUTPUT_ROOT / "session.json").write_text(
    json.dumps(session, indent=2) + "\n", encoding="utf-8"
)
print("Saved results", OUTPUT_ROOT)

The saved files can be downloaded with `kaggle kernels output` after the remote
run completes. The results are evidence only after the saved source version,
data identity and any shared run link have been checked. GPU training and energy
measurement need separate implementations and have not been demonstrated here.